# NDVI Dataset Expansion using Google Earth Engine
## Smart Contract-Based Automated Crop Insurance System

---

**Purpose:** Replace synthetic NDVI data with real satellite data

**Data Source:** Google Earth Engine (MODIS & Sentinel-2)
- Real satellite imagery
- 2000-2023 coverage
- 10+ agricultural regions

**Time Required:** ~20-30 minutes

---
## Step 1: Setup Google Earth Engine

**First Time Setup:**
1. Go to: https://earthengine.google.com/signup/
2. Sign up with Google account
3. Wait for approval (usually instant)
4. Come back and run cells below

In [ ]:
# Install Earth Engine API
!pip install earthengine-api pandas numpy matplotlib

In [ ]:
import ee
import pandas as pd
import numpy as np
import time
from datetime import datetime

print("="*70)
print("NDVI DATASET EXPANSION MODULE")
print("="*70)
print("✓ Libraries imported")

In [ ]:
# Authenticate (FIRST TIME ONLY)
# This will open a browser window for authentication
try:
    ee.Authenticate()
    print("✓ Authentication successful")
except:
    print("ℹ️  Already authenticated or using service account")

In [ ]:
# Initialize Earth Engine
ee.Initialize()
print("✓ Google Earth Engine initialized successfully!")

---
## Step 2: Define Agricultural Regions in India

We'll collect NDVI data from major agricultural regions

In [ ]:
# Define bounding boxes for major agricultural regions
agricultural_regions = {
    # North India (Major wheat belt)
    'Punjab': ee.Geometry.Rectangle([74.5, 29.5, 76.5, 32.5]),
    'Haryana': ee.Geometry.Rectangle([74.5, 27.5, 77.5, 30.5]),
    'Western_UP': ee.Geometry.Rectangle([77.0, 27.0, 80.0, 30.0]),
    'Eastern_UP': ee.Geometry.Rectangle([80.0, 25.0, 84.0, 27.5]),
    
    # East India
    'Bihar': ee.Geometry.Rectangle([84.0, 24.5, 88.0, 27.0]),
    'West_Bengal': ee.Geometry.Rectangle([86.0, 22.0, 89.5, 27.0]),
    
    # Central India
    'Madhya_Pradesh': ee.Geometry.Rectangle([74.0, 21.0, 82.0, 26.5]),
    'Maharashtra': ee.Geometry.Rectangle([73.0, 16.0, 80.0, 22.0]),
    
    # South India
    'Karnataka': ee.Geometry.Rectangle([74.0, 12.0, 78.5, 18.5]),
    'Andhra_Pradesh': ee.Geometry.Rectangle([77.0, 13.0, 84.5, 19.5]),
    'Tamil_Nadu': ee.Geometry.Rectangle([76.5, 8.0, 80.5, 13.5]),
    
    # West India
    'Gujarat': ee.Geometry.Rectangle([68.0, 20.0, 74.5, 24.5]),
}

print(f"✓ Defined {len(agricultural_regions)} agricultural regions")
print("\nRegions:")
for region in agricultural_regions.keys():
    print(f"  • {region}")

---
## Step 3: Function to Extract MODIS NDVI Time Series

In [ ]:
def get_modis_ndvi_timeseries(region_name, geometry, start_date='2000-01-01', end_date='2023-12-31'):
    """
    Extract MODIS NDVI time series for a region
    
    Parameters:
    -----------
    region_name : str
        Name of the region
    geometry : ee.Geometry
        Bounding box of the region
    start_date : str
        Start date (YYYY-MM-DD)
    end_date : str
        End date (YYYY-MM-DD)
    
    Returns:
    --------
    pandas.DataFrame
        NDVI time series data
    """
    
    # Load MODIS NDVI collection (16-day composite, 250m resolution)
    collection = ee.ImageCollection('MODIS/061/MOD13Q1') \
        .filterBounds(geometry) \
        .filterDate(start_date, end_date) \
        .select('NDVI')
    
    # Function to extract NDVI value for each image
    def extract_ndvi(image):
        # Get mean NDVI for the region
        stats = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geometry,
            scale=250,  # 250m resolution
            maxPixels=1e9
        )
        
        # Create feature with date and NDVI
        return ee.Feature(None, {
            'date': image.date().format('YYYY-MM-dd'),
            'ndvi': stats.get('NDVI'),
            'region': region_name
        })
    
    # Map extraction function over collection
    features = collection.map(extract_ndvi)
    
    # Convert to DataFrame
    try:
        data = features.getInfo()
        df = pd.DataFrame([f['properties'] for f in data['features']])
        
        if not df.empty:
            # Scale NDVI (stored as integers in Earth Engine)
            df['ndvi'] = pd.to_numeric(df['ndvi'], errors='coerce')
            df['ndvi'] = df['ndvi'] / 10000  # Scale to 0-1 range
            df['ndvi'] = df['ndvi'].clip(0, 1)
            
            # Convert date
            df['date'] = pd.to_datetime(df['date'])
            df['year'] = df['date'].dt.year
            df['month'] = df['date'].dt.month
            
            # Remove any rows with missing NDVI
            df = df.dropna(subset=['ndvi'])
        
        return df
    
    except Exception as e:
        print(f"  ❌ Error for {region_name}: {e}")
        return pd.DataFrame()

print("✓ MODIS extraction function defined")

---
## Step 4: Extract NDVI Data for All Regions

**⚠️ Note:** This takes 10-20 minutes

In [ ]:
all_ndvi_data = []

print("="*70)
print("STARTING NDVI DATA EXTRACTION")
print("="*70)
print(f"\nRegions to process: {len(agricultural_regions)}")
print(f"Time period: 2000-2023")
print(f"Estimated time: 10-20 minutes\n")
print("-"*70)

start_time = time.time()

for idx, (region_name, geometry) in enumerate(agricultural_regions.items(), 1):
    print(f"\n[{idx}/{len(agricultural_regions)}] Processing: {region_name}")
    
    try:
        df_region = get_modis_ndvi_timeseries(region_name, geometry, '2000-01-01', '2023-12-31')
        
        if not df_region.empty:
            all_ndvi_data.append(df_region)
            print(f"  ✓ Success: {len(df_region)} NDVI records")
            print(f"    Date range: {df_region['date'].min().date()} to {df_region['date'].max().date()}")
            print(f"    NDVI range: {df_region['ndvi'].min():.3f} to {df_region['ndvi'].max():.3f}")
        else:
            print(f"  ⚠️  No data retrieved")
    
    except Exception as e:
        print(f"  ❌ Error: {e}")
    
    # Small pause between regions
    time.sleep(1)

end_time = time.time()
elapsed_time = (end_time - start_time) / 60

print("\n" + "="*70)
print("EXTRACTION COMPLETE")
print("="*70)
print(f"Time taken: {elapsed_time:.1f} minutes")
print(f"Successful regions: {len(all_ndvi_data)}/{len(agricultural_regions)}")
print("="*70)

---
## Step 5: Combine and Analyze NDVI Data

In [ ]:
# Combine all regional data
df_ndvi_real = pd.concat(all_ndvi_data, ignore_index=True)

print("="*70)
print("REAL SATELLITE NDVI DATA SUMMARY")
print("="*70)
print(f"\nTotal Records: {len(df_ndvi_real):,}")
print(f"Regions: {df_ndvi_real['region'].nunique()}")
print(f"Date Range: {df_ndvi_real['date'].min().date()} to {df_ndvi_real['date'].max().date()}")
print(f"Years Covered: {df_ndvi_real['year'].max() - df_ndvi_real['year'].min() + 1}")
print(f"\nNDVI Statistics:")
print(f"  Mean: {df_ndvi_real['ndvi'].mean():.3f}")
print(f"  Std:  {df_ndvi_real['ndvi'].std():.3f}")
print(f"  Min:  {df_ndvi_real['ndvi'].min():.3f}")
print(f"  Max:  {df_ndvi_real['ndvi'].max():.3f}")
print("\n" + "="*70)

In [ ]:
# Display first rows
print("\nFirst 10 records:")
df_ndvi_real.head(10)

In [ ]:
# Check data by region
print("\nRecords per Region:")
region_counts = df_ndvi_real.groupby('region').size().sort_values(ascending=False)
for region, count in region_counts.items():
    print(f"  {region:20} : {count:>5} records")

---
## Step 6: Create Time Series Samples for LSTM Training

In [ ]:
def create_ndvi_sequences(df, region, sequence_length=120, step=30):
    """
    Create NDVI sequences for LSTM training
    
    Parameters:
    -----------
    df : DataFrame
        NDVI data
    region : str
        Region name
    sequence_length : int
        Length of each sequence (default: 120 = ~5 years of 16-day data)
    step : int
        Step size for sliding window
    
    Returns:
    --------
    list of dict
        List of sequences with metadata
    """
    # Filter for region and sort by date
    region_data = df[df['region'] == region].sort_values('date').reset_index(drop=True)
    
    sequences = []
    
    # Create sliding windows
    for i in range(0, len(region_data) - sequence_length + 1, step):
        sequence = region_data.iloc[i:i+sequence_length]
        
        # Calculate sequence characteristics
        mean_ndvi = sequence['ndvi'].mean()
        trend = sequence['ndvi'].iloc[-1] - sequence['ndvi'].iloc[0]
        
        # Classify scenario based on patterns
        if trend < -0.15:  # Significant drop
            scenario = 'stress'
        elif mean_ndvi > 0.6:  # High NDVI
            scenario = 'healthy'
        elif mean_ndvi < 0.4:  # Low NDVI
            scenario = 'poor'
        else:
            scenario = 'normal'
        
        sequences.append({
            'region': region,
            'start_date': sequence['date'].iloc[0],
            'end_date': sequence['date'].iloc[-1],
            'ndvi_series': sequence['ndvi'].values,
            'mean_ndvi': mean_ndvi,
            'trend': trend,
            'scenario': scenario
        })
    
    return sequences

print("✓ Sequence creation function defined")

In [ ]:
# Create sequences for all regions
all_sequences = []

print("Creating NDVI sequences for LSTM training...\n")

for region in df_ndvi_real['region'].unique():
    region_sequences = create_ndvi_sequences(df_ndvi_real, region, sequence_length=120, step=30)
    all_sequences.extend(region_sequences)
    print(f"  {region:20} : {len(region_sequences):>4} sequences")

print(f"\nTotal sequences created: {len(all_sequences):,}")

# Convert to DataFrame
df_sequences = pd.DataFrame([{
    'region': s['region'],
    'start_date': s['start_date'],
    'end_date': s['end_date'],
    'mean_ndvi': s['mean_ndvi'],
    'trend': s['trend'],
    'scenario': s['scenario']
} for s in all_sequences])

print("\nScenario Distribution:")
print(df_sequences['scenario'].value_counts())

---
## Step 7: Save NDVI Data

In [ ]:
# Save raw NDVI time series
df_ndvi_real.to_csv('ndvi_satellite_data_real.csv', index=False)
print("✓ Saved: ndvi_satellite_data_real.csv")
print(f"  Records: {len(df_ndvi_real):,}")

In [ ]:
# Save sequences metadata
df_sequences.to_csv('ndvi_sequences_metadata.csv', index=False)
print("✓ Saved: ndvi_sequences_metadata.csv")
print(f"  Sequences: {len(df_sequences):,}")

In [ ]:
# Save complete sequences (for direct use in LSTM)
import pickle

with open('ndvi_sequences_complete.pkl', 'wb') as f:
    pickle.dump(all_sequences, f)

print("✓ Saved: ndvi_sequences_complete.pkl")
print(f"  Contains: {len(all_sequences):,} complete NDVI sequences")
print(f"  Ready for LSTM training!")

---
## Step 8: Visualize NDVI Data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: NDVI over time for sample region
ax1 = axes[0, 0]
sample_region = df_ndvi_real['region'].iloc[0]
region_data = df_ndvi_real[df_ndvi_real['region'] == sample_region].sort_values('date')
ax1.plot(region_data['date'], region_data['ndvi'], linewidth=1, alpha=0.7)
ax1.set_title(f'NDVI Time Series: {sample_region}', fontweight='bold')
ax1.set_xlabel('Date')
ax1.set_ylabel('NDVI')
ax1.grid(True, alpha=0.3)

# Plot 2: NDVI distribution
ax2 = axes[0, 1]
ax2.hist(df_ndvi_real['ndvi'], bins=50, color='green', edgecolor='black', alpha=0.7)
ax2.set_title('NDVI Distribution (All Regions)', fontweight='bold')
ax2.set_xlabel('NDVI Value')
ax2.set_ylabel('Frequency')
ax2.axvline(0.6, color='red', linestyle='--', label='Healthy threshold')
ax2.axvline(0.3, color='orange', linestyle='--', label='Stress threshold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Plot 3: NDVI by region (boxplot)
ax3 = axes[1, 0]
region_order = df_ndvi_real.groupby('region')['ndvi'].median().sort_values(ascending=False).index
sns.boxplot(data=df_ndvi_real, y='region', x='ndvi', order=region_order, ax=ax3)
ax3.set_title('NDVI Distribution by Region', fontweight='bold')
ax3.set_xlabel('NDVI')
ax3.set_ylabel('')

# Plot 4: Seasonal pattern (average NDVI by month)
ax4 = axes[1, 1]
monthly_avg = df_ndvi_real.groupby('month')['ndvi'].mean()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax4.plot(monthly_avg.index, monthly_avg.values, marker='o', linewidth=2, markersize=8, color='darkgreen')
ax4.set_title('Average NDVI by Month', fontweight='bold')
ax4.set_xlabel('Month')
ax4.set_ylabel('Average NDVI')
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(months, rotation=45)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ndvi_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: ndvi_analysis.png")
plt.show()

---
## Step 9: Compare with Synthetic Data

In [ ]:
print("="*70)
print("SYNTHETIC vs REAL NDVI DATA COMPARISON")
print("="*70)
print("\nSYNTHETIC DATA (Original):")
print(f"  Samples: 200 (50 per scenario)")
print(f"  Source: Mathematically generated")
print(f"  Realism: Low")
print(f"  Credibility: Limited")
print("\nREAL SATELLITE DATA (New):")
print(f"  Records: {len(df_ndvi_real):,}")
print(f"  Sequences: {len(all_sequences):,}")
print(f"  Source: MODIS satellite (NASA)")
print(f"  Regions: {df_ndvi_real['region'].nunique()}")
print(f"  Time span: {df_ndvi_real['year'].max() - df_ndvi_real['year'].min() + 1} years")
print(f"  Realism: Very High")
print(f"  Credibility: Excellent")
print("\n" + "="*70)
print(f"\nIMPROVEMENT: {len(all_sequences)/200:.1f}x more data!")
print("="*70)

---
## Step 10: Usage in LSTM Training

In [ ]:
# Example: How to use this data in your NDVI LSTM notebook

print("HOW TO USE IN YOUR LSTM NOTEBOOK:")
print("="*70)
print("\nOption 1: Load sequences directly")
print("""
import pickle
with open('ndvi_sequences_complete.pkl', 'rb') as f:
    real_sequences = pickle.load(f)

# Extract NDVI series for training
X_data = [seq['ndvi_series'] for seq in real_sequences]
scenarios = [seq['scenario'] for seq in real_sequences]
""")

print("\nOption 2: Load raw data and create custom sequences")
print("""
df_ndvi = pd.read_csv('ndvi_satellite_data_real.csv')

# Filter for specific region
punjab_data = df_ndvi[df_ndvi['region'] == 'Punjab']
ndvi_series = punjab_data['ndvi'].values

# Use for LSTM training
""")
print("="*70)

---
## Summary

In [ ]:
print("\n" + "="*70)
print("NDVI DATASET EXPANSION COMPLETE!")
print("="*70)
print("\n📊 SUMMARY:")
print("─"*70)
print(f"Synthetic Data (Original): 200 samples")
print(f"Real Satellite Data (New): {len(df_ndvi_real):,} records")
print(f"NDVI Sequences Created:    {len(all_sequences):,}")
print(f"Improvement Factor:        {len(all_sequences)/200:.0f}x")
print("─"*70)
print("\n📁 GENERATED FILES:")
print("  1. ndvi_satellite_data_real.csv    (Raw NDVI data)")
print("  2. ndvi_sequences_metadata.csv     (Sequence info)")
print("  3. ndvi_sequences_complete.pkl     (Complete sequences) ⭐")
print("  4. ndvi_analysis.png               (Visualization)")
print("\n✅ NEXT STEP:")
print("  In your NDVI_Analysis_LSTM.ipynb, replace synthetic data with:")
print('    import pickle')
print('    with open("ndvi_sequences_complete.pkl", "rb") as f:')
print('        all_ndvi_data = pickle.load(f)')
print("\n  Then use real sequences for LSTM training!")
print("="*70)